In [3]:
import pickle
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
import os, json, time

LABEL_MAP = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
LABELS    = ["World", "Sports", "Business", "Sci/Tech"]

# Load raw text data — Pipeline needs raw text, not pre-transformed matrices
# This is the key difference: Pipeline handles transformation internally
train_df = pd.read_parquet("../data/train.parquet")
test_df  = pd.read_parquet("../data/test.parquet")

y_train = train_df["label"].values
y_test  = test_df["label"].values

print(f"Train samples : {len(train_df)}")
print(f"Test samples  : {len(test_df)}")
print(f"\nSample text   : {train_df['text'].iloc[0][:100]}")
print(f"Label         : {LABEL_MAP[y_train[0]]}")

# Build the Pipeline — two named steps
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=10000,
        ngram_range=(1, 2),
        sublinear_tf=True
    )),
    ("classifier", LogisticRegression(
        C=1.0,
        solver="lbfgs",
        multi_class="multinomial",
        max_iter=1000,
        random_state=42,
        n_jobs=-1
    ))
])


print(f"\nPipeline steps:")
for name, step in pipeline.steps:
    print(f"  [{name}] {step.__class__.__name__}")

print("\nPipeline built successfully.")
print("Ready for Block 2 — training.")

Train samples : 120000
Test samples  : 7600

Sample text   : Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\b
Label         : Business

Pipeline steps:
  [tfidf] TfidfVectorizer
  [classifier] LogisticRegression

Pipeline built successfully.
Ready for Block 2 — training.


In [4]:
# Train the entire pipeline in one call
# fit() internally calls:
#   1. tfidf.fit_transform(X_train)  — learns vocab, transforms text
#   2. classifier.fit(X_tfidf, y_train) — trains LR on the matrix
# You never touch the intermediate matrix yourself

print("\nTraining pipeline...")
start = time.time()

pipeline.fit(train_df["text"], y_train)

elapsed = time.time() - start
print(f"Training complete in {elapsed:.1f}s")

# Predict — pipeline calls tfidf.transform() then classifier.predict()
# Raw text in, label out. One line.
y_pred = pipeline.predict(test_df["text"])

# Probability scores — confidence per class
y_prob = pipeline.predict_proba(test_df["text"])

# Core metrics
accuracy  = (y_pred == y_test).mean()
macro_f1  = f1_score(y_test, y_pred, average="macro")

print(f"\nPipeline results:")
print(f"  Accuracy  : {accuracy:.4f}")
print(f"  Macro F1  : {macro_f1:.4f}")

# Load your Day 3 baseline from the experiment log
with open("../model/artifacts/experiment_log.json", "r") as f:
    baseline = json.load(f)

baseline_f1 = baseline["macro_f1"]
diff        = macro_f1 - baseline_f1

print(f"\nBaseline F1 (Day 3 LR)  : {baseline_f1:.4f}")
print(f"Pipeline F1 (Day 4)     : {macro_f1:.4f}")
print(f"Difference              : {diff:+.4f}")

if abs(diff) < 0.001:
    print("\nResult: Numbers match — Pipeline is correctly reproducing Day 3 results.")
    print("This confirms the Pipeline is not changing the model, just wrapping it.")
else:
    print("\nResult: Small difference detected — likely due to solver randomness.")
    print("Check that random_state=42 is set in both models.")

print("\nFull classification report:")
print(classification_report(y_test, y_pred, target_names=LABELS, digits=4))


Training pipeline...


d:\Ai\aienv\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Training complete in 42.3s

Pipeline results:
  Accuracy  : 0.9100
  Macro F1  : 0.9099

Baseline F1 (Day 3 LR)  : 0.9099
Pipeline F1 (Day 4)     : 0.9099
Difference              : -0.0000

Result: Numbers match — Pipeline is correctly reproducing Day 3 results.
This confirms the Pipeline is not changing the model, just wrapping it.

Full classification report:
              precision    recall  f1-score   support

       World     0.9286    0.9037    0.9160      1900
      Sports     0.9506    0.9732    0.9618      1900
    Business     0.8775    0.8747    0.8761      1900
    Sci/Tech     0.8828    0.8884    0.8856      1900

    accuracy                         0.9100      7600
   macro avg     0.9099    0.9100    0.9099      7600
weighted avg     0.9099    0.9100    0.9099      7600



In [5]:
from sklearn.model_selection import GridSearchCV

# These are the combinations GridSearchCV will try
# 3 values of max_features × 3 values of C = 9 combinations
# Each combination is trained and evaluated 5 times (cv=5 folds)
# Total = 9 × 5 = 45 training runs — all automatic

param_grid = {
    "tfidf__max_features" : [5000, 10000, 20000],
    "classifier__C"       : [0.1, 1.0, 10.0]
}

print("Running GridSearchCV (45 fits)...")
print("This will take 3-6 minutes on CPU — expected behaviour.\n")

grid_search = GridSearchCV(
    estimator  = pipeline,
    param_grid = param_grid,
    cv         = 5,           # 5-fold cross validation
    scoring    = "f1_macro",  # metric to optimise
    n_jobs     = -1,          # use all CPU cores
    verbose    = 1            # print progress
)

start = time.time()
grid_search.fit(train_df["text"], y_train)
elapsed = time.time() - start

print(f"\nGrid search complete in {elapsed:.1f}s")
print(f"Best parameters : {grid_search.best_params_}")
print(f"Best CV F1      : {grid_search.best_score_:.4f}")

Running GridSearchCV (45 fits)...
This will take 3-6 minutes on CPU — expected behaviour.

Fitting 5 folds for each of 9 candidates, totalling 45 fits


d:\Ai\aienv\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



Grid search complete in 353.4s
Best parameters : {'classifier__C': 1.0, 'tfidf__max_features': 20000}
Best CV F1      : 0.9046


In [7]:
# grid_search.best_estimator_ is the winning pipeline
# already retrained on full training data automatically
best_pipeline = grid_search.best_estimator_

# Final evaluation on test set — first and only time test data is used
y_pred_best = best_pipeline.predict(test_df["text"])
y_prob_best = best_pipeline.predict_proba(test_df["text"])
results={
  "model": "LogisticRegression",
  "vectorizer": "TF-IDF (10k features, bigrams, sublinear_tf)",
  "train_samples": 120000,
  "test_samples": 7600,
  "accuracy": 0.91,
  "macro_f1": 0.9099,
  "per_class_f1": {
    "World": 0.916,
    "Sports": 0.9618,
    "Business": 0.8761,
    "Sci/Tech": 0.8856
  },
  "training_time_s": 18.33
}

final_f1       = f1_score(y_test, y_pred_best, average="macro")
baseline_f1    = results["macro_f1"]   # your Day 3 JSON log
improvement    = final_f1 - baseline_f1

print("="*55)
print("Final evaluation — best pipeline on held-out test set")
print("="*55)
print(f"\nDay 3 baseline F1  : {baseline_f1:.4f}")
print(f"Day 4 best F1      : {final_f1:.4f}")
print(f"Improvement        : {improvement:+.4f}")
print(f"\nClassification report:")
print(classification_report(y_test, y_pred_best, target_names=LABELS, digits=4))

# Save the best pipeline as single artifact
os.makedirs("../model/artifacts", exist_ok=True)
with open("../model/artifacts/best_pipeline.pkl", "wb") as f:
    pickle.dump(best_pipeline, f)

print("Saved best_pipeline.pkl")
print(f"File size : {os.path.getsize('../model/artifacts/best_pipeline.pkl') / 1024 / 1024:.1f} MB")

# Update experiment log
experiment = {
    "model"             : "Pipeline(TF-IDF + LogisticRegression)",
    "best_params"       : grid_search.best_params_,
    "cv_f1_macro"       : round(float(grid_search.best_score_), 4),
    "test_f1_macro"     : round(float(final_f1), 4),
    "baseline_f1"       : baseline_f1,
    "improvement"       : round(float(improvement), 4),
    "train_samples"     : len(train_df),
    "test_samples"      : len(test_df),
}

with open("../model/artifacts/experiment_log.json", "w") as f:
    json.dump(experiment, f, indent=2)

print("\nExperiment log updated.")
print(json.dumps(experiment, indent=2))

Final evaluation — best pipeline on held-out test set

Day 3 baseline F1  : 0.9099
Day 4 best F1      : 0.9175
Improvement        : +0.0076

Classification report:
              precision    recall  f1-score   support

       World     0.9328    0.9053    0.9188      1900
      Sports     0.9533    0.9774    0.9652      1900
    Business     0.8899    0.8853    0.8876      1900
    Sci/Tech     0.8942    0.9026    0.8984      1900

    accuracy                         0.9176      7600
   macro avg     0.9175    0.9176    0.9175      7600
weighted avg     0.9175    0.9176    0.9175      7600

Saved best_pipeline.pkl
File size : 1.3 MB

Experiment log updated.
{
  "model": "Pipeline(TF-IDF + LogisticRegression)",
  "best_params": {
    "classifier__C": 1.0,
    "tfidf__max_features": 20000
  },
  "cv_f1_macro": 0.9046,
  "test_f1_macro": 0.9175,
  "baseline_f1": 0.9099,
  "improvement": 0.0076,
  "train_samples": 120000,
  "test_samples": 7600
}


In [8]:
# Test the saved pipeline on raw text — simulating a real API call
# This is exactly what your FastAPI endpoint will do on Day 7

with open("../model/artifacts/best_pipeline.pkl", "rb") as f:
    loaded_pipeline = pickle.load(f)

# Simulate 4 real API requests — one per class
test_inputs = [
    "The Federal Reserve raised interest rates by 25 basis points amid inflation concerns",
    "Manchester United secured a dramatic late winner in the Champions League semifinal",
    "The prime minister announced new sanctions against the regime following border violations",
    "Researchers unveiled a new large language model surpassing GPT-4 on key benchmarks"
]

expected = ["Business", "Sports", "World", "Sci/Tech"]

print("End-to-end prediction test:")
print("="*65)

all_correct = True
for text, expected_label in zip(test_inputs, expected):
    prediction  = loaded_pipeline.predict([text])[0]
    proba       = loaded_pipeline.predict_proba([text])[0]
    confidence  = proba.max()
    predicted_label = LABEL_MAP[prediction]
    correct     = predicted_label == expected_label

    if not correct:
        all_correct = False

    status = "PASS" if correct else "FAIL"
    print(f"\n[{status}] {text[:60]}...")
    print(f"  Expected   : {expected_label}")
    print(f"  Predicted  : {predicted_label}")
    print(f"  Confidence : {confidence:.4f}")
    print(f"  All probs  : { {LABEL_MAP[i]: round(float(p),4) for i,p in enumerate(proba)} }")

print("\n" + "="*65)
print(f"Result: {'All 4 predictions correct' if all_correct else 'Some predictions failed — check above'}")
print("\nPipeline is ready for FastAPI serving on Day 7.")

End-to-end prediction test:

[PASS] The Federal Reserve raised interest rates by 25 basis points...
  Expected   : Business
  Predicted  : Business
  Confidence : 0.9139
  All probs  : {'World': 0.065, 'Sports': 0.0127, 'Business': 0.9139, 'Sci/Tech': 0.0084}

[PASS] Manchester United secured a dramatic late winner in the Cham...
  Expected   : Sports
  Predicted  : Sports
  Confidence : 0.9928
  All probs  : {'World': 0.0048, 'Sports': 0.9928, 'Business': 0.0016, 'Sci/Tech': 0.0007}

[PASS] The prime minister announced new sanctions against the regim...
  Expected   : World
  Predicted  : World
  Confidence : 0.8898
  All probs  : {'World': 0.8898, 'Sports': 0.0133, 'Business': 0.053, 'Sci/Tech': 0.0439}

[PASS] Researchers unveiled a new large language model surpassing G...
  Expected   : Sci/Tech
  Predicted  : Sci/Tech
  Confidence : 0.9400
  All probs  : {'World': 0.0442, 'Sports': 0.0042, 'Business': 0.0116, 'Sci/Tech': 0.94}

Result: All 4 predictions correct

Pipeline is ready 